# RSNA Knee: первый бейзлайн (локальная версия)

Первая версия пайплайна: одна картинка из 3 срезов на исследование, DINOv2 ViT-B, BCE, 5 фолдов.
Раньше картинки читались из DICOM на Kaggle, теперь из того же локального «сырого» кэша, что и в
`02_report_labels_mil.ipynb`: по файлу на исследование со всеми срезами серий. Логика обучения, модель и обработка меток не менялись.

**Что нужно в `data/raw/`:** `studies/<ID>.npz` (кэш из `03_build_cache_kaggle.ipynb`) и `train.csv`.
**Веса ViT-B:** `weights/vit_base_patch14_dinov2.pth` рядом с ноутбуком.

**Известная проблема этой версии** (из-за неё val AUC около 0.5): у 4349 из 4407 исследований метки
в `train.csv` пустые, а `np.nan_to_num(labels, nan=0.0)` делает их нулями, то есть «здоров по всем 12
находкам». Модели не на чем учиться. Исправление - во второй версии: метки из текстов заключений.

### Imports

In [ ]:
import os
import warnings
import random
import copy
from pathlib import Path

import pandas as pd
import numpy as np

import sklearn
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold

import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from torchinfo import summary

import timm

import torch
from torch import nn
from torch .utils.data import DataLoader, Dataset
import torch.nn.functional as F
from torchvision import transforms
from torchvision.transforms import v2


warnings.filterwarnings("ignore")
print("All imported.")

In [ ]:
device = torch.device("mps"
if torch.backends.mps.is_available()
else "cuda"
if torch.cuda.is_available()
else "cpu")

print(f"Device: '{device}'")

In [ ]:
SEED = 2026
N_SPLITS = 5
BATCH_SIZE = 8
NUM_EPOCHS = 10
LR = 2e-5

def seed_everything(seed=42):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    if torch.backends.mps.is_available():
        torch.mps.manual_seed(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic=(True)
    torch.backends.cudnn.benchmark=(False)
    os.environ["PYTHONHASHSEED"] = str(seed)

seed_everything(SEED)
print(f"General seed: {SEED}")

In [ ]:
# Ноутбук запускается из папки rsna-knee-abnormality-detection
DATA_DIR = Path("data") / "raw"            # studies/ + train.csv
STUDY_DIR = DATA_DIR / "studies"           # по файлу на исследование: {"slot0": [n_срезов, 224, 224], ...}
WEIGHTS_PATH = Path("weights") / "vit_base_patch14_dinov2.pth"   # предобученные веса DINOv2 ViT-B
OUT_DIR = Path("outputs") / "baseline"     # сюда пишутся лучшие веса каждого фолда
OUT_DIR.mkdir(parents=True, exist_ok=True)

for p in (STUDY_DIR, DATA_DIR / "train.csv", WEIGHTS_PATH):
    assert p.exists(), f"нет {p}"

In [ ]:
TARGET_COLS = [
    'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus',
    'Medial OA', 'Lateral OA', 'PF OA', 'Effusion',
    'Synovitis', "Baker's", 'Contusion', 'Fracture'
]

In [ ]:
print(f"studies/: {len(list(STUDY_DIR.glob('*.npz')))} файлов")
for p in sorted(DATA_DIR.iterdir()):
    if p.is_file():
        print(f"{p.name:<20} {p.stat().st_size / 1024 ** 2:>10.1f} МБ")

### Data

In [ ]:
train_df = pd.read_csv(DATA_DIR / "train.csv")
study_ids = train_df["StudyInstanceUID"].astype(str).tolist()

# Какую картинку берёт бейзлайн: одна серия и три её среза вокруг центра стека.
# Слоты в файле: slot0 SAG_FS, slot1 COR_FS, slot2 AX_FS, slot3 SAG_NOFS, slot4 COR_NOFS, slot5 AX_NOFS.
SLOT_ORDER = ["slot0", "slot3"]   # сагиттальная с подавлением жира; если её нет - сагиттальная без него
STEP = 4                          # как в первой версии: центральный срез и два среза на 4 дальше в обе стороны


def load_three_slices(study_id):
    """[3, H, W] uint8: срезы c-STEP, c, c+STEP из сагиттальной серии исследования (c - центр стека)."""
    with np.load(STUDY_DIR / f"{study_id}.npz") as z:
        key = next((k for k in SLOT_ORDER if k in z.files), sorted(z.files)[0])
        stack = z[key]                                        # [n_срезов, H, W], по порядку в теле
    c = len(stack) // 2
    idx = np.clip([c - STEP, c, c + STEP], 0, len(stack) - 1)
    return stack[idx]

In [ ]:
study = random.choice(study_ids)
image = load_three_slices(study)   # [3, 224, 224]: три среза -> три канала

print(f"Shape image: {image.shape}")
print(f"Type data image: {image.dtype}")

fig, axes = plt.subplots(1, 3, figsize=(9, 3))
for ax, ch, name in zip(axes, image, ["R: c-STEP", "G: центр", "B: c+STEP"]):
    ax.imshow(ch, cmap="gray"); ax.set_title(name); ax.axis("off")
plt.show()

### Custom Dataset

In [ ]:
class RSNAKneeDataset(Dataset):
    """Одна картинка на исследование: три среза сагиттальной серии вокруг центра стека.

    __getitem__ сам открывает файл исследования, берёт три среза (load_three_slices) и делит на 255.
    Обработка меток не менялась (вместе с проблемой NaN -> 0).
    """

    def __init__(self, df, label_smoothing=0.5):
        self.df = df.reset_index(drop=True)
        self.label_smoothing = label_smoothing

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = load_three_slices(str(row["StudyInstanceUID"])).astype(np.float32) / 255.0
        image = torch.tensor(image, dtype=torch.float32)          # [3, 224, 224]

        labels = row[TARGET_COLS].values.astype(np.float32)
        labels = np.nan_to_num(labels, nan=0.0)
        labels = torch.tensor(labels, dtype=torch.float32)

        if self.label_smoothing > 0:
            labels = labels * (1.0 - self.label_smoothing) + 0.5 * self.label_smoothing
        return image, labels

### Model

In [ ]:
# class RSNA_efficientnet_b0(nn.Module):
#   def __init__(self, model_name="efficientnet_b0", pretrained=False):
#     super().__init__()

#     self.model = timm.create_model(
#         model_name, pretrained=pretrained, num_classes=len(TARGET_COLS)
#     )

#     weights_path = (
#         "/kaggle/input/datasets/tihon2010/weights-efficientnet-b0/efficientnet_b0.pth"
#     )
#     if os.path.exists(weights_path):
#       print("Download weights ...")
#       state_dict = torch.load(weights_path, map_location="cpu")

#       if "classifier.weight" in state_dict:
#         del state_dict["classifier.weight"]
#       if "classifier.bias" in state_dict:
#         del state_dict["classifier.bias"]

#       self.model.load_state_dict(state_dict, strict=False)
#     else:
#       print("Weights not found path:", weights_path)

#   def forward(self, x):
#     return self.model(x)

In [ ]:
# class RSNA_convnext_small(nn.Module):
#   def __init__(self, model_name="convnext_small", pretrained=False):
#     super().__init__()

#     self.model = timm.create_model(
#         model_name, pretrained=pretrained, num_classes=len(TARGET_COLS)
#     )

#     weights_path = (
#         "/kaggle/input/datasets/tihon2010/weights-convnext-small/convnext_small.pth"
#     )
#     if os.path.exists(weights_path):
#       print("Download weights ...")
#       state_dict = torch.load(weights_path, map_location="cpu")

#       for key in list(state_dict.keys()):
#           if "head" in key or "classifier" in key:
#               del state_dict[key]

#       self.model.load_state_dict(state_dict, strict=False)
#     else:
#       print("Weights not found path:", weights_path)

#   def forward(self, x):
#     return self.model(x)

In [ ]:
class RSNA_dinov2(nn.Module):
    def __init__(self, model_name="vit_base_patch14_dinov2.lvd142m", pretrained=False):
        super().__init__()

        self.model = timm.create_model(
            model_name, pretrained=pretrained, num_classes=len(TARGET_COLS), dynamic_img_size=True
        )

        weights_path = WEIGHTS_PATH
        if os.path.exists(weights_path):
            print("Download weights ...")
            state_dict = torch.load(weights_path, map_location="cpu")

            for key in list(state_dict.keys()):
                if "head" in key or "classifier" in key:
                    del state_dict[key]

            self.model.load_state_dict(state_dict, strict=False)
        else:
            print("Weights not found path:", weights_path)

    def forward(self, x):
        return self.model(x)

### Train step

In [ ]:
def train_one_epoch(model, dataloader, optimizer, criterion, device):

    model.train()
    running_loss = 0.0

    for images, labels in dataloader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    return running_loss / len(dataloader)

### Valid step

In [ ]:
def validate_one_epoch(model, dataloader, criterion, device):

    model.eval()
    running_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item()

            preds = torch.sigmoid(outputs).cpu().numpy()
            all_preds.append(preds)
            all_labels.append(labels.cpu().numpy())

    val_loss = running_loss / len(dataloader)
    all_preds = np.concatenate(all_preds, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)

    class_aucs = []
    for i in range(all_labels.shape[1]):
        if len(np.unique(all_labels[:, i])) > 1:
            try:
                score = roc_auc_score(all_labels[:, i], all_preds[:, i])
                if not np.isnan(score):
                    class_aucs.append(score)
            except Exception:
                pass

    macro_auc = np.mean(class_aucs) if class_aucs else 0.0

    return val_loss, macro_auc

### Train and Eval

In [ ]:
# train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))

# gkf = GroupKFold(n_splits=N_SPLITS)
# train_df["fold"] = -1

# for fold, (train_idx, val_idx) in enumerate(gkf.split(train_df, groups=train_df["StudyInstanceUID"])):
#     train_df.loc[val_idx, "fold"] = fold

# df_train = train_df[train_df["fold"] != 0].reset_index(drop=True)
# df_val = train_df[train_df["fold"] == 0].reset_index(drop=True)
# print(f"Train patients: {len(df_train)}, Val patients: {len(df_val)}")

# train_dataset = RSNAKneeDataset(df_train, img_size=224, is_train=True)
# val_dataset = RSNAKneeDataset(df_val, img_size=224, is_train=True)

# train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
# val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# model = RSNAKneeModel(model_name="efficientnet_b0", pretrained=False).to(device)
# optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
# criterion = nn.BCEWithLogitsLoss()
# scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

# best_val_auc = 0.0

# print(f"Start train model ...")
# for epoch in range(NUM_EPOCHS):
#     train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
#     val_loss, val_auc = validate_one_epoch(model, val_loader, criterion, device)

#     scheduler.step()

#     print(f"Epoch {epoch+1}/{NUM_EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Macro-AUC: {val_auc:.4f}")

#     if val_auc > best_val_auc:
#         best_val_auc = val_auc
#         torch.save(model.state_dict(), "best_rsna_model.pth")
#         print("Saved new best model (best_rsna_model.pth)")

# print("Train finish.")

In [ ]:

gkf = GroupKFold(n_splits=N_SPLITS)
train_df["fold"] = -1

for fold, (train_idx, val_idx) in enumerate(gkf.split(train_df, groups=train_df["StudyInstanceUID"])):
    train_df.loc[val_idx, "fold"] = fold

full_train_dataset = RSNAKneeDataset(train_df, label_smoothing=0.05)
full_val_dataset = RSNAKneeDataset(train_df, label_smoothing=0.0)

best_auc_score = []

print(f"\n=== Starting {N_SPLITS}-Fold Training ===")
for fold in range(N_SPLITS):
    print(f"\n--- Fold {fold+1} ---")

    train_idx = train_df[train_df["fold"] != fold].index.tolist()
    val_idx = train_df[train_df["fold"] == fold].index.tolist()

    train_dataset = torch.utils.data.Subset(full_train_dataset, train_idx)
    val_dataset = torch.utils.data.Subset(full_val_dataset, val_idx)

    # num_workers=0: класс объявлен в ноутбуке, а воркеры на macOS его не видят (в 02 класс вынесен в файл)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    model = RSNA_dinov2("vit_base_patch14_dinov2.lvd142m", pretrained=False).to(device)

    for name, param in model.named_parameters():
        param.requires_grad = False
        if "head" in name or "blocks.10." in name or "blocks.11." in name:
            param.requires_grad = True

    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=1e-5,
        weight_decay=1e-2
    )

    criterion = torch.nn.BCEWithLogitsLoss()
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

    best_val_auc = 0.0
    model_save_path = OUT_DIR / f"best_rsna_dinov2_model_fold_{fold+1}.pth"

    for epoch in range(NUM_EPOCHS):
        train_loss = train_one_epoch(model=model,
                                     dataloader=train_loader,
                                     optimizer=optimizer,
                                     criterion=criterion,
                                     device=device
                                    )
        val_loss, val_auc = validate_one_epoch(model=model,
                                         dataloader=val_loader,
                                         criterion=criterion,
                                         device=device
                                         )
        scheduler.step()

        print(f"Epoch {epoch+1}/{NUM_EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Auc: {val_auc:.4f}")

        if val_auc > best_val_auc:
            best_val_auc = val_auc
            torch.save(model.state_dict(), model_save_path)
            print(f"Saved new best model for fold {fold+1} ({model_save_path})")

    best_auc_score.append(best_val_auc)

print(f"\n=== Training finished ===")
print(f'Mean Best Val Macro-AUC across all folds: {np.mean(best_auc_score):.4f}')

### Predict

Предсказание test в этой версии убрано: test DICOM есть только на Kaggle, а локальный кэш содержит
только train. Сабмит для второй версии делает `04_kaggle_submission.ipynb`.